In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("/home/pcsl/Documents/plecs/sepic/plecs_python_auto/AImodel")
import model_create as mc
from keras import optimizers
from sklearn.preprocessing import MinMaxScaler, StandardScaler


df = pd.read_csv('/home/pcsl/Documents/plecs/sepic/plecs_python_auto/out/sepic_data/inner_loop/ripple_results.csv')
df = df[df["mosfet_cond_loss"] <= 1000] # maximum limitation
df = df[df["L2"] >= 0.00007] # maximum limitation
df = df[df["C1"] >= 0.000005] # maximum limitation

print(f"행 개수 :  {len(df)}")

2026-04-30 15:07:28.312238: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777529248.323606 1097256 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777529248.326948 1097256 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-30 15:07:28.339927: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.




Failed to import TF-Keras. Please note that TF-Keras is not installed by default when you install TensorFlow Probability. This is so that JAX-only users do not have to install TensorFlow or TF-Keras. To use TensorFlow Probability with TensorFlow, please install the tf-keras or tf-keras-nightly package.
This can be be done through installing the tensorflow-probability[tf] extra.




ModuleNotFoundError: No module named 'tf_keras'

In [ ]:


X = df[['L1', 'L2', 'C1', 'C2']].values

X = mc.generate_features(X)
print(f"X[0] = {X[0]}")
print(f"len(X[0]) = {len(X[0])}")
y_ripple  = df[['i_L1_ripple_rate', 'V_out_ripple_rate']].values
y_mos_loss = df[['mosfet_cond_loss', 'mosfet_switch_loss']].values

splits = mc.split_data(X, y_ripple, y_mos_loss)

X_train, X_val, X_test = splits[0]
y_ripple_train, y_ripple_val, y_ripple_test = splits[1]
y_mos_loss_train, y_mos_loss_val, y_mos_loss_test = splits[2]


# 학습용
X_train, X_val, X_test, x_scaler = mc.scale_data(
    X_train, X_val, X_test,
    scaler = StandardScaler()
)

#ripple
y_ripple_train, y_ripple_val, y_ripple_test, y_ripple_scaler = mc.scale_data(
    y_ripple_train, y_ripple_val, y_ripple_test,
    scaler = StandardScaler()
)

# mosfet loss
# y_mos_loss_train, y_mos_loss_val, y_mos_loss_test, y_mos_loss_scaler = mc.scale_data(
#     y_mos_loss_train, y_mos_loss_val, y_mos_loss_test,
#     scaler = StandardScaler()
# )
# print("var: ",y_mos_loss_scaler.var_, "\nmean:", y_mos_loss_scaler.mean_, "\nstd:", y_mos_loss_scaler.scale_)
# print(y_mos_loss_scaler) # mosfet_switch_loss minimum is 13


y_mos_loss_train = np.log1p(y_mos_loss_train)
y_mos_loss_val   = np.log1p(y_mos_loss_val)
y_mos_loss_test  = np.log1p(y_mos_loss_test)

model_mos_loss2 =  mc.build_bnn_residual(
    input_dim=4,
    output_dim=2,
    dataset_size=len(X_train)
)

model_mos_loss= mc.build_test_residual(input_dim=len(X[0]), output_dim=2, model_name="mos_loss")



history_mosfet1 = mc.train_and_evaluate_sklearn(
    model_mos_loss2,
    X_train, y_mos_loss_train,
    X_val,   y_mos_loss_val,
    X_test,  y_mos_loss_test,
    model_name='model_mos_loss2 Model',
)

history_mosfet2 = mc.train_and_evaluate(
    model_mos_loss,
    X_train, y_mos_loss_train,
    X_val,   y_mos_loss_val,
    X_test,  y_mos_loss_test,
    model_name='Mosfet Loss Model',
    epochs=1000, batch_size=64
)

X[0] = [4.00000000e-04 1.00000000e-05 1.00000000e-06 1.00000000e-05
 2.49993750e+03 9.99000999e+04 9.90099010e+05 9.99000999e+04
 4.00000000e-10 1.00000000e-10 9.61538462e+07 9.90099010e+07
 3.99600400e+01 9.99000999e-02]
len(X[0]) = 14


AttributeError: module 'model_create' has no attribute 'build_bnn_residual'

In [ ]:
preds = np.array([
    model_mos_loss2(X_test, training=True)  # sampling
    for _ in range(100)
])

mean = preds.mean(axis=0)
std = preds.std(axis=0)

In [ ]:
# example =np.asarray([[0.000900, 1.29990e-05, 9.67904e-06, 0.000188]])
# example = mc.generate_features(example)
# example =  x_scaler.transform(example)

# non_scaled = model_mos_loss.predict(example)

# # y_pred = y_mos_loss_scaler.inverse_transform(non_scaled)      
# y_pred = np.expm1(non_scaled)

# print(non_scaled)
# print(y_pred)



# non_scaled = model_mos_loss2.predict(example)

# # y_pred = y_mos_loss_scaler.inverse_transform(non_scaled)      
# y_pred = np.expm1(non_scaled)

# print(y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
[[2.8420513 1.7250279]]
[[16.150911   4.6126776]]
[[16.16942856  3.56427497]]


In [ ]:
y = model_mos_loss.predict(np.array(X_test))

from sklearn.metrics import mean_squared_error, mean_absolute_error

y_pred_orig = np.expm1(y)[:, 0]
y_true_orig = np.expm1(y_mos_loss_test)[:, 0]


abs_err = np.abs(y_true_orig - y_pred_orig)

idx = np.argmax(abs_err)

print("worst index (filtered):", idx)
print("X_test:", x_scaler.inverse_transform([X_test[idx]]))
print("y_true:", y_true_orig[idx])
print("y_pred:", y_pred_orig[idx])

mse = mean_squared_error(y_true_orig, y_pred_orig)
mae = mean_absolute_error(y_true_orig, y_pred_orig)

print("MSE:", mse)
print("MAE:", mae)


46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
worst index (filtered): 451
X_test: [[1.20000000e-03 1.00000000e-05 2.00000000e-06 1.90000000e-04
  8.33326389e+02 9.99000999e+04 4.97512438e+05 5.26288090e+03
  2.40000000e-09 1.90000000e-09 8.06451613e+07 8.40336134e+07
  1.19880120e+02 1.05257618e-02]]
y_true: 785.6191708468907
y_pred: 108.6607
MSE: 2041.9793821118114
MAE: 8.682974833649574


In [ ]:
print("y_pred max:", np.max(y[:,0]))
print("y_pred min:", np.min(y[:,0]))

print("y_pred max:", np.max(y_pred_orig[:,0]))
print("y_pred min:", np.min(y_pred_orig[:,0]))

print("inf in y_pred:", np.isinf(y_pred).any())
print("inf in y_pred_orig:", np.isinf(y_pred_orig).any())

y_pred max: 6.3174205
y_pred min: 2.534407


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed